In [1]:
!pip install langchain langchain_community langchain-huggingface pypdf chromadb
!pip install "unstructured[all-docs]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/22

In [4]:
from google.colab import userdata
o = userdata.get('HUGG')

In [5]:
import os
os.environ['HF_TOKEN'] = o

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEndpoint

In [7]:
doc = PyPDFLoader("/content/HR-Policies-Manuals.pdf")
document = doc.load()

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", " "], chunk_size=500, chunk_overlap=10)
chunks = text_splitter.split_documents(document)

In [9]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings,persist_directory="db")
vectorstore.persist()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipython-input-1080215134.py:5: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [10]:
retriver = vectorstore.as_retriever(search_kwargs = {"k":3})

In [11]:
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough,RunnableSequence

In [12]:
def f1(data):
  l = []
  for i in data:
    l.append(i.page_content)
  return "\n\n".join(l)

In [13]:
r2 = RunnableLambda(f1)

In [14]:
chain1 = RunnableSequence(retriver,r2)

In [15]:
from langchain_core.prompts import ChatPromptTemplate,HumanMessagePromptTemplate,SystemMessagePromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser

In [16]:
cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("""you are a helpful HR who is having 5 years of experience."""),
                                       HumanMessagePromptTemplate.from_template("answer the quention on below context provided context:{context} query:{query}")])

In [17]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

repo_id = "mistralai/Mistral-7B-Instruct-v0.2"

llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    max_new_tokens=512,
    repetition_penalty=1.03,
    huggingfacehub_api_token=os.environ['HF_TOKEN']
)

model = ChatHuggingFace(llm=llm)

In [18]:
otp = StrOutputParser()

In [19]:
chain2 = RunnableParallel({"context": chain1, "query": RunnablePassthrough()})

In [20]:
chain3 = RunnableSequence(chain2,cpt,model,otp)

In [21]:
RAG_pipeline = chain2 | cpt | model | otp

In [23]:
RAG_pipeline.invoke("how many leaves i can get in a month")

"Based on the **Leave Policy** mentioned in your context, the number of **paid leaves** an employee can avail in a month depends on their **Date of Joining (DoJ)**. Here's how it works:\n\n### **Paid Leaves per Month (Pro-Rata Basis)**\n- **1st to 7th of every month:** **2 full days** of paid leave\n- **8th to 14th of every month:** **1.5 days** of paid leave\n- **15th to 21st of every month:** **1 day** of paid leave\n- **22nd to 31st of every month:** **0.5 days** of paid leave\n\n### **Example Scenarios**\n- If you joined on **2nd May**, you are eligible for **2 full days** of paid leave in **May**.\n- If you joined on **9th June**, you get **1.5 days** of paid leave in **June**.\n- If you joined on **16th July**, only **1 full day** is allowed in **July**.\n- If you joined on **28th August**, you are eligible for **0.5 days** of paid leave in **August**.\n\n### **Additional Notes**\n- The leaves **cannot be carried forward** to the next month unless you adjust them within the same 